# TACO + AquaTrash YOLO26n Segmentation Training Pipeline

This standalone Kaggle notebook downloads TACO from its `annotations.json` file, downloads the referenced TACO images into `/kaggle/working`, reads AquaTrash directly from `/kaggle/input`, merges both datasets into one 8-class YOLO segmentation dataset, visualizes labels, tunes YOLO26n-seg with Optuna, tracks experiments with MLflow, trains the final best model, evaluates it on validation/test splits, and exports the best artifacts.

Expected inputs:

- TACO: either no Kaggle input is needed, because `annotations.json` is downloaded from Hugging Face, or you can attach a dataset containing only `annotations.json`. Images are downloaded from the URLs inside that annotation file.
- AquaTrash segmentations: `karimaouaouda/aquatrash-segmentations` or the metadata spelling `karimaouaouda/aquatrash-segmantations`, containing `labels_final.json`.
- AquaTrash raw images/boxes: `harshpanwar/aquatrash`, containing `annotations.csv` and `Images/`.


In [ ]:
# 0) Install dependencies if missing
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = [
    ("ultralytics", "ultralytics>=8.0"),
    ("mlflow", "mlflow>=2.14"),
    ("optuna", "optuna>=4.0"),
    ("yaml", "pyyaml>=6.0"),
    ("PIL", "Pillow>=10.0"),
]

missing = [pkg for module, pkg in REQUIRED_PACKAGES if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")

In [ ]:
# 1) Imports and paths
from __future__ import annotations

import csv
import json
import math
import os
import random
import shutil
import time
from io import BytesIO
from urllib.request import urlretrieve
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import requests
import matplotlib.patches as patches
import mlflow
import optuna
import pandas as pd
import yaml
from PIL import Image
from ultralytics import YOLO

IS_KAGGLE = Path("/kaggle").exists()
INPUT_DIR = Path("/kaggle/input") if IS_KAGGLE else Path("../data/raw")
INPUT_DIRS = [INPUT_DIR]
if not IS_KAGGLE:
    for extra_input_dir in [Path("data/raw"), Path("../data/raw")]:
        if extra_input_dir not in INPUT_DIRS:
            INPUT_DIRS.append(extra_input_dir)
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./working")

DATA_DIR = WORKING_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_TACO_DIR = RAW_DIR / "taco"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROCESSED_DIR / "taco_aquatrash_yolo_seg"
RUNS_DIR = WORKING_DIR / "runs" / "yolo-seg"
ARTIFACTS_DIR = WORKING_DIR / "artifacts" / "yolo26n-merged"
MLRUNS_DIR = WORKING_DIR / "mlruns"

for p in [DATA_DIR, RAW_DIR, RAW_TACO_DIR, PROCESSED_DIR, OUTPUT_DIR, RUNS_DIR, ARTIFACTS_DIR, MLRUNS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.10
TEST_RATIO = 0.20
random.seed(SEED)

ANNOTATIONS_PATH = DATA_DIR / "annotations.json"
DATASET_JSON_URL = "https://huggingface.co/datasets/karimaouaouda/taco/resolve/main/annotations.json"

print("IS_KAGGLE:", IS_KAGGLE)
print("INPUT_DIR:", INPUT_DIR)
print("INPUT_DIRS:", INPUT_DIRS)
print("WORKING_DIR:", WORKING_DIR)

In [ ]:
# 2) Training configuration
# Override these in Kaggle with environment variables if needed.
YOLO_MODEL = os.getenv("YOLO_MODEL", "yolo26n-seg.pt")
EPOCHS = int(os.getenv("EPOCHS", "50"))
IMGSZ = int(os.getenv("IMGSZ", "640"))
BATCH = int(os.getenv("BATCH", "16"))
DEVICE = os.getenv("DEVICE", "0")
PATIENCE = int(os.getenv("PATIENCE", "20"))
OPTUNA_TRIALS = int(os.getenv("OPTUNA_TRIALS", "10"))
TUNE_EPOCHS = int(os.getenv("TUNE_EPOCHS", "12"))
SKIP_OPTUNA = os.getenv("SKIP_OPTUNA", "0") == "1"

EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "waste-seg-yolo26n-taco-aquatrash")
RUN_NAME = os.getenv("RUN_NAME", "yolo26n-seg-taco-aquatrash")

TARGET_LABELS = [
    "organic_waste",
    "plastic_bottle",
    "plastic_bag",
    "rigid_plastic",
    "metal_can",
    "glass",
    "paper_cardboard",
    "mixed_waste",
]
LABEL_TO_ID = {name: idx for idx, name in enumerate(TARGET_LABELS)}

print({
    "YOLO_MODEL": YOLO_MODEL,
    "EPOCHS": EPOCHS,
    "IMGSZ": IMGSZ,
    "BATCH": BATCH,
    "DEVICE": DEVICE,
    "OPTUNA_TRIALS": OPTUNA_TRIALS,
    "TUNE_EPOCHS": TUNE_EPOCHS,
    "SKIP_OPTUNA": SKIP_OPTUNA,
})

In [ ]:
# 3) Download TACO annotations/images, then discover AquaTrash files

def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def find_kaggle_annotations_json(input_dir: Path) -> Path | None:
    """Prefer a lightweight attached annotations.json if the user provides one."""
    candidates = []
    for root in INPUT_DIRS:
        if root.exists():
            candidates.extend(root.rglob("annotations.json"))
    scored = []
    for path in candidates:
        try:
            payload = load_json(path)
            images = payload.get("images", [])
            anns = payload.get("annotations", [])
            cats = payload.get("categories", [])
            score = len(images) + len(anns) + len(cats) * 100
            scored.append((score, path, len(images), len(anns), len(cats)))
        except Exception:
            continue
    if not scored:
        return None
    scored.sort(reverse=True, key=lambda row: row[0])
    print("Using attached annotations.json candidate:", scored[0])
    return scored[0][1]


def prepare_taco_annotations() -> Path:
    if ANNOTATIONS_PATH.exists():
        print("TACO annotations already exist:", ANNOTATIONS_PATH)
        return ANNOTATIONS_PATH

    attached_annotations = find_kaggle_annotations_json(INPUT_DIR)
    if attached_annotations is not None:
        shutil.copy2(attached_annotations, ANNOTATIONS_PATH)
        print("Copied TACO annotations from Kaggle input:", attached_annotations)
    else:
        print("Downloading TACO annotations.json from Hugging Face...")
        urlretrieve(DATASET_JSON_URL, ANNOTATIONS_PATH)

        print("Downloaded TACO annotations to:", ANNOTATIONS_PATH)
    return ANNOTATIONS_PATH


def download_taco_images(annotations_path: Path, output_dir: Path, show_progress: bool = True) -> int:
    
    # check if there is a local dataset installed with images already present, if so skip downloading and just count them
    # this isn't meant to kaggle, only for local runs where the user might have already downloaded the TACO dataset manually to save time and bandwidth

    if not IS_KAGGLE:
        local_taco_root = Path("data/raw/taco")
        if local_taco_root.exists():
            print("Found existing TACO images in data/raw/taco/images, skipping download.")
            # dataset structure is expected to be data/raw/taco/batch_xxx/xxx.jpg, so we need to count all jpg files under data/raw/taco
            existing_images = list(local_taco_root.rglob("*.jpg"))

            # copy the folder to working dir
            if not output_dir.exists():
                shutil.copytree(local_taco_root, output_dir)

            print(f"Found {len(existing_images)} existing TACO images under data/raw/taco.")
            return len(existing_images)
        else:
            print("No existing TACO images found in data/raw/taco/images, proceeding to download.")

    
    payload = load_json(annotations_path)
    images = payload.get("images", [])
    output_dir.mkdir(parents=True, exist_ok=True)

    downloaded = 0
    existing = 0
    failed = []

    for idx, image in enumerate(images, start=1):
        rel_path = Path(image["file_name"])
        dst = output_dir / rel_path
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            existing += 1
            continue

        urls = [image.get("flickr_url"), image.get("flickr_640_url"), image.get("coco_url")]
        urls = [url for url in urls if url]
        success = False
        for url in urls:
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                img = Image.open(BytesIO(response.content)).convert("RGB")
                img.save(dst)
                downloaded += 1
                success = True
                break
            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
        if not success:
            failed.append({"file_name": image["file_name"], "urls": urls, "error": last_error if urls else "no_url"})

        if show_progress and idx % 100 == 0:
            print(f"TACO images checked {idx}/{len(images)} | downloaded={downloaded} existing={existing} failed={len(failed)}")

    summary = {
        "referenced_images": len(images),
        "downloaded": downloaded,
        "existing": existing,
        "failed": len(failed),
        "failed_examples": failed[:20],
        "output_dir": str(output_dir),
    }
    (DATA_DIR / "taco_download_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2)[:2000])
    return len(images) - len(failed)


def existing_input_dir(*names: str) -> Path | None:
    """Return the first input directory matching one of the expected names."""
    for root in INPUT_DIRS:
        for name in names:
            candidate = root / name
            if candidate.exists():
                return candidate
    return None


def first_existing_file(root: Path, pattern: str) -> Path | None:
    matches = list(root.rglob(pattern)) if root.exists() else []
    return matches[0] if matches else None


def find_aquatrash_files(input_dir: Path) -> tuple[Path, Path, Path]:
    """Find AquaTrash files using the dataset sources from kernel-metadata.json.

    Kaggle dataset source mapping:
    - karimaouaouda/aquatrash-segmentations, or metadata typo
      karimaouaouda/aquatrash-segmantations: labels_final.json
    - harshpanwar/aquatrash: annotations.csv and Images/
    """
    segmentation_root = existing_input_dir(
        "aquatrash-segmentations",
        "aquatrash-segmantations",
        "karimaouaouda-aquatrash-segmentations",
        "karimaouaouda-aquatrash-segmantations",
    )
    aquatrash_root = existing_input_dir(
        "aquatrash",
        "harshpanwar-aquatrash",
    )

    if segmentation_root is None:
        print("Expected segmentation dataset folder was not found; falling back to recursive labels_final.json search.")
        labels_candidates = [p for root in INPUT_DIRS if root.exists() for p in root.rglob("labels_final.json")]
        if not labels_candidates:
            raise FileNotFoundError(
                "Could not find labels_final.json. Attach karimaouaouda/aquatrash-segmentations "
                "or karimaouaouda/aquatrash-segmantations to the Kaggle notebook."
            )
        labels_path = labels_candidates[0]
        segmentation_root = labels_path.parent
    else:
        labels_path = first_existing_file(segmentation_root, "labels_final.json")
        if labels_path is None:
            raise FileNotFoundError(f"labels_final.json was not found inside {segmentation_root}")

    if aquatrash_root is None:
        print("Expected AquaTrash dataset folder was not found; falling back to recursive annotations.csv search.")
        csv_candidates = [p for root in INPUT_DIRS if root.exists() for p in root.rglob("annotations.csv") if "segment" not in str(p).lower() and "segmant" not in str(p).lower()]
        if not csv_candidates:
            raise FileNotFoundError(
                "Could not find annotations.csv. Attach harshpanwar/aquatrash to the Kaggle notebook."
            )
        csv_path = csv_candidates[0]
        aquatrash_root = csv_path.parent
    else:
        csv_path = first_existing_file(aquatrash_root, "annotations.csv")
        if csv_path is None:
            raise FileNotFoundError(f"annotations.csv was not found inside {aquatrash_root}")

    image_dir_candidates = [
        aquatrash_root / "Images",
        aquatrash_root / "images",
    ]
    image_dir_candidates += [
        path for path in aquatrash_root.rglob("*") if path.is_dir() and path.name.lower() == "images"
    ]
    images_dir = next((path for path in image_dir_candidates if path.exists()), None)
    if images_dir is None:
        raise FileNotFoundError(f"Could not find an Images folder inside {aquatrash_root}")

    print("AquaTrash segmentation dataset root:", segmentation_root)
    print("AquaTrash raw image/box dataset root:", aquatrash_root)
    return labels_path, csv_path, images_dir


TACO_ANNOTATIONS_PATH = prepare_taco_annotations()
TACO_IMAGES_ROOT = RAW_TACO_DIR
available_taco_images = download_taco_images(TACO_ANNOTATIONS_PATH, TACO_IMAGES_ROOT, show_progress=True)
AQUA_LABELS_PATH, AQUA_BOXES_CSV_PATH, AQUA_IMAGES_ROOT = find_aquatrash_files(INPUT_DIR)

print("TACO_ANNOTATIONS_PATH:", TACO_ANNOTATIONS_PATH)
print("TACO_IMAGES_ROOT:", TACO_IMAGES_ROOT)
print("available_taco_images:", available_taco_images)
print("AQUA_LABELS_PATH:", AQUA_LABELS_PATH)
print("AQUA_BOXES_CSV_PATH:", AQUA_BOXES_CSV_PATH)
print("AQUA_IMAGES_ROOT:", AQUA_IMAGES_ROOT)


In [ ]:
# 4) Class remapping
OLD_TO_NEW_LABEL = {
    "Food waste": "organic_waste",
    "Other plastic bottle": "plastic_bottle",
    "Clear plastic bottle": "plastic_bottle",
    "Plastic film": "plastic_bag",
    "Six pack rings": "plastic_bag",
    "Garbage bag": "plastic_bag",
    "Other plastic wrapper": "plastic_bag",
    "Single-use carrier bag": "plastic_bag",
    "Polypropylene bag": "plastic_bag",
    "Crisp packet": "plastic_bag",
    "Plastic bottle cap": "rigid_plastic",
    "Plastic lid": "rigid_plastic",
    "Other plastic": "rigid_plastic",
    "Disposable plastic cup": "rigid_plastic",
    "Foam cup": "rigid_plastic",
    "Other plastic cup": "rigid_plastic",
    "Spread tub": "rigid_plastic",
    "Tupperware": "rigid_plastic",
    "Disposable food container": "rigid_plastic",
    "Foam food container": "rigid_plastic",
    "Other plastic container": "rigid_plastic",
    "Plastic glooves": "rigid_plastic",
    "Plastic utensils": "rigid_plastic",
    "Squeezable tube": "rigid_plastic",
    "Plastic straw": "rigid_plastic",
    "Styrofoam piece": "mixed_waste",
    "Aluminium foil": "metal_can",
    "Aluminium blister pack": "metal_can",
    "Metal bottle cap": "metal_can",
    "Food Can": "metal_can",
    "Aerosol": "metal_can",
    "Drink can": "metal_can",
    "Metal lid": "metal_can",
    "Pop tab": "metal_can",
    "Scrap metal": "metal_can",
    "Glass bottle": "glass",
    "Broken glass": "glass",
    "Glass cup": "glass",
    "Glass jar": "glass",
    "Toilet tube": "paper_cardboard",
    "Other carton": "paper_cardboard",
    "Egg carton": "paper_cardboard",
    "Drink carton": "paper_cardboard",
    "Corrugated carton": "paper_cardboard",
    "Meal carton": "paper_cardboard",
    "Pizza box": "paper_cardboard",
    "Paper cup": "paper_cardboard",
    "Magazine paper": "paper_cardboard",
    "Tissues": "paper_cardboard",
    "Wrapping paper": "paper_cardboard",
    "Normal paper": "paper_cardboard",
    "Paper bag": "paper_cardboard",
    "Plastified paper bag": "paper_cardboard",
    "Paper straw": "paper_cardboard",
    "Battery": "mixed_waste",
    "Carded blister pack": "mixed_waste",
    "Rope & strings": "mixed_waste",
    "Shoe": "mixed_waste",
    "Unlabeled litter": "mixed_waste",
    "Cigarette": "mixed_waste",
}

AQUATRASH_LABEL_MAP = {
    "glass": "glass",
    "metal": "metal_can",
    "metal_can": "metal_can",
    "paper": "paper_cardboard",
    "paper_cardboard": "paper_cardboard",
    "plastic": "rigid_plastic",
    "plastic_battle": "plastic_bottle",
    "plastic_bottle": "plastic_bottle",
    "plastic_bag": "plastic_bag",
    "rigid_plastic": "rigid_plastic",
    "organic_waste": "organic_waste",
    "mixed_waste": "mixed_waste",
}


def resolve_taco_label(category: dict[str, Any]) -> str:
    name = str(category.get("name", "")).strip()
    supercategory = str(category.get("supercategory", "")).strip()
    lower_name = name.lower()
    lower_supercategory = supercategory.lower()

    if name in OLD_TO_NEW_LABEL:
        return OLD_TO_NEW_LABEL[name]
    if lower_supercategory == "bottle":
        return "glass" if "glass" in lower_name else "plastic_bottle"
    if lower_supercategory == "bottle cap":
        return "metal_can" if "metal" in lower_name else "rigid_plastic"
    if lower_supercategory in {"paper", "carton", "paper bag"}:
        return "paper_cardboard"
    if lower_supercategory == "plastic bag & wrapper":
        return "plastic_bag"
    if lower_supercategory == "plastic container":
        return "rigid_plastic"
    if lower_supercategory == "cup":
        if lower_name.startswith("glass"):
            return "glass"
        if lower_name.startswith("paper"):
            return "paper_cardboard"
        return "rigid_plastic"
    if lower_supercategory == "lid":
        return "metal_can" if "metal" in lower_name else "rigid_plastic"
    if lower_supercategory == "other plastic":
        return "rigid_plastic"
    if lower_supercategory == "straw":
        return "paper_cardboard" if lower_name.startswith("paper") else "rigid_plastic"
    if lower_supercategory == "food waste":
        return "organic_waste"
    if lower_supercategory in {"battery", "unlabeled litter", "rope & strings", "shoe", "cigarette"}:
        return "mixed_waste"
    if "glass" in lower_name:
        return "glass"
    if "can" in lower_name or "foil" in lower_name or "metal" in lower_name:
        return "metal_can"
    if "paper" in lower_name or "carton" in lower_name or "tissue" in lower_name or "box" in lower_name:
        return "paper_cardboard"
    if any(t in lower_name for t in ["plastic", "foam", "tupperware", "tube", "glove", "utensil", "straw"]):
        return "rigid_plastic"
    return "mixed_waste"

In [ ]:
# 5) Merge raw annotations
def normalize_polygon(seg: list[float], width: int, height: int) -> list[str]:
    points = []
    for i in range(0, len(seg), 2):
        if i + 1 >= len(seg):
            break
        x = max(0.0, min(1.0, float(seg[i]) / float(width)))
        y = max(0.0, min(1.0, float(seg[i + 1]) / float(height)))
        points.extend([f"{x:.6f}", f"{y:.6f}"])
    return points


def split_by_source(records: list[dict[str, Any]]) -> dict[int, str]:
    by_source = defaultdict(list)
    for rec in records:
        by_source[rec["source"]].append(rec["id"])

    image_to_split = {}
    for source, ids in sorted(by_source.items()):
        ids = list(ids)
        random.Random(SEED).shuffle(ids)
        n = len(ids)
        train_end = int(n * TRAIN_RATIO)
        val_end = train_end + int(n * VAL_RATIO)
        for image_id in ids[:train_end]:
            image_to_split[image_id] = "train"
        for image_id in ids[train_end:val_end]:
            image_to_split[image_id] = "val"
        for image_id in ids[val_end:]:
            image_to_split[image_id] = "test"
    return image_to_split


def build_merged_records():
    images = []
    annotations_by_image = defaultdict(list)
    summary = {"classes": TARGET_LABELS, "sources": {}, "skipped_annotations": 0, "aquatrash_csv_box_rows": 0}
    next_image_id = 1
    next_ann_id = 1

    taco = load_json(TACO_ANNOTATIONS_PATH)
    taco_categories = {int(c["id"]): c for c in taco.get("categories", [])}
    taco_source_to_merged = {}
    taco_counts = Counter()

    for image in taco.get("images", []):
        src = TACO_IMAGES_ROOT / image["file_name"]
        if not src.exists():
            continue
        merged_id = next_image_id
        next_image_id += 1
        taco_source_to_merged[int(image["id"])] = merged_id
        images.append({
            "id": merged_id,
            "source": "taco",
            "source_file_name": image["file_name"],
            "file_name": (Path("taco") / image["file_name"]).as_posix(),
            "source_path": src,
            "width": int(image["width"]),
            "height": int(image["height"]),
        })

    for ann in taco.get("annotations", []):
        merged_image_id = taco_source_to_merged.get(int(ann["image_id"]))
        category = taco_categories.get(int(ann["category_id"]))
        if merged_image_id is None or category is None:
            summary["skipped_annotations"] += 1
            continue
        label = resolve_taco_label(category)
        annotations_by_image[merged_image_id].append({
            "id": next_ann_id,
            "source": "taco",
            "category_id": LABEL_TO_ID[label],
            "segmentation": ann.get("segmentation", []),
            "bbox": ann.get("bbox"),
        })
        next_ann_id += 1
        taco_counts[label] += 1

    aqua = load_json(AQUA_LABELS_PATH)
    aqua_categories = {int(c["id"]): str(c["name"]) for c in aqua.get("categories", [])}
    aqua_source_to_file = {int(img["id"]): img["file_name"] for img in aqua.get("images", [])}
    aqua_file_to_merged = {}
    aqua_counts = Counter()

    for image in aqua.get("images", []):
        file_name = image["file_name"]
        src = AQUA_IMAGES_ROOT / file_name
        if not src.exists():
            continue
        merged_id = next_image_id
        next_image_id += 1
        aqua_file_to_merged[file_name] = merged_id
        images.append({
            "id": merged_id,
            "source": "aquatrash",
            "source_file_name": file_name,
            "file_name": (Path("aquatrash") / file_name).as_posix(),
            "source_path": src,
            "width": int(image["width"]),
            "height": int(image["height"]),
        })

    for ann in aqua.get("annotations", []):
        file_name = aqua_source_to_file.get(int(ann["image_id"]))
        merged_image_id = aqua_file_to_merged.get(file_name)
        if merged_image_id is None:
            summary["skipped_annotations"] += 1
            continue
        source_label = aqua_categories.get(int(ann["category_id"]), "mixed_waste")
        label = AQUATRASH_LABEL_MAP.get(source_label, "mixed_waste")
        annotations_by_image[merged_image_id].append({
            "id": next_ann_id,
            "source": "aquatrash",
            "category_id": LABEL_TO_ID[label],
            "segmentation": ann.get("segmentation", []),
            "bbox": ann.get("bbox"),
        })
        next_ann_id += 1
        aqua_counts[label] += 1

    aqua_csv_box_counts = Counter()
    aqua_csv_files = set()
    if AQUA_BOXES_CSV_PATH.exists():
        with AQUA_BOXES_CSV_PATH.open("r", encoding="utf-8", newline="") as handle:
            for row in csv.DictReader(handle):
                summary["aquatrash_csv_box_rows"] += 1
                aqua_csv_files.add(row["image_name"])
                csv_label = AQUATRASH_LABEL_MAP.get(str(row["class_name"]).strip(), "mixed_waste")
                aqua_csv_box_counts[csv_label] += 1

    summary["sources"]["taco"] = {
        "images": sum(1 for img in images if img["source"] == "taco"),
        "annotations": int(sum(taco_counts.values())),
        "counts_by_class": dict(taco_counts),
    }
    summary["sources"]["aquatrash"] = {
        "images": sum(1 for img in images if img["source"] == "aquatrash"),
        "annotations": int(sum(aqua_counts.values())),
        "counts_by_class": dict(aqua_counts),
        "csv_box_files": len(aqua_csv_files),
        "csv_box_counts_by_class": dict(aqua_csv_box_counts),
        "segmentation_source": str(AQUA_LABELS_PATH),
        "box_source": str(AQUA_BOXES_CSV_PATH),
    }
    summary["total_images"] = len(images)
    summary["total_annotations"] = sum(len(v) for v in annotations_by_image.values())
    return images, annotations_by_image, summary


merged_images, merged_annotations_by_image, merge_summary = build_merged_records()
print(json.dumps(merge_summary, indent=2))

In [ ]:
# 6) Build YOLO segmentation dataset
def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def build_yolo_dataset(images, annotations_by_image, output_dir: Path):
    reset_dir(output_dir)
    for split in ["train", "val", "test"]:
        (output_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (output_dir / "labels" / split).mkdir(parents=True, exist_ok=True)

    image_to_split = split_by_source(images)
    split_summary = {}
    categories = [{"id": i, "name": name, "supercategory": name} for i, name in enumerate(TARGET_LABELS)]
    split_payloads = {split: {"images": [], "annotations": [], "categories": categories} for split in ["train", "val", "test"]}

    for image in images:
        split = image_to_split[image["id"]]
        dst_image = output_dir / "images" / split / image["file_name"]
        dst_label = (output_dir / "labels" / split / image["file_name"]).with_suffix(".txt")
        dst_image.parent.mkdir(parents=True, exist_ok=True)
        dst_label.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(image["source_path"], dst_image)

        label_lines = []
        for ann in annotations_by_image.get(image["id"], []):
            segs = ann.get("segmentation", [])
            if not isinstance(segs, list):
                continue
            for seg in segs:
                if not isinstance(seg, list) or len(seg) < 6:
                    continue
                points = normalize_polygon(seg, image["width"], image["height"])
                if len(points) >= 6:
                    label_lines.append(f"{ann['category_id']} " + " ".join(points))
            split_payloads[split]["annotations"].append({
                "id": ann["id"],
                "image_id": image["id"],
                "category_id": ann["category_id"],
                "segmentation": segs,
                "bbox": ann.get("bbox"),
                "source": ann.get("source"),
            })

        dst_label.write_text("\n".join(label_lines), encoding="utf-8")
        split_payloads[split]["images"].append({
            "id": image["id"],
            "width": image["width"],
            "height": image["height"],
            "file_name": image["file_name"],
            "source": image["source"],
            "source_file_name": image["source_file_name"],
        })

    for split in ["train", "val", "test"]:
        (output_dir / f"annotations_{split}.json").write_text(json.dumps(split_payloads[split], indent=2), encoding="utf-8")
        label_paths = list((output_dir / "labels" / split).rglob("*.txt"))
        split_summary[split] = {
            "images": len(split_payloads[split]["images"]),
            "annotations": len(split_payloads[split]["annotations"]),
            "label_files": len(label_paths),
            "instances": sum(1 for p in label_paths for line in p.read_text(encoding="utf-8").splitlines() if line.strip()),
            "images_by_source": dict(Counter(img["source"] for img in split_payloads[split]["images"])),
        }

    dataset_yaml = output_dir / "dataset.yaml"
    dataset_yaml.write_text(
        yaml.safe_dump({
            "path": output_dir.resolve().as_posix(),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "nc": len(TARGET_LABELS),
            "names": {i: name for i, name in enumerate(TARGET_LABELS)},
        }, sort_keys=False),
        encoding="utf-8",
    )

    full_summary = {
        **merge_summary,
        "dataset_yaml": str(dataset_yaml.resolve()),
        "splits": split_summary,
        "output_dir": str(output_dir.resolve()),
        "split_ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    }
    (output_dir / "merge_summary.json").write_text(json.dumps(full_summary, indent=2), encoding="utf-8")
    return dataset_yaml, full_summary


dataset_yaml_path, dataset_summary = build_yolo_dataset(merged_images, merged_annotations_by_image, OUTPUT_DIR)
print("dataset_yaml_path:", dataset_yaml_path)
print(json.dumps(dataset_summary["splits"], indent=2))

In [ ]:
# 7) Validate YOLO labels
class_counts = Counter()
bad_files = []
empty_files = []

for split in ["train", "val", "test"]:
    for label_path in (OUTPUT_DIR / "labels" / split).rglob("*.txt"):
        text = label_path.read_text(encoding="utf-8").strip()
        if not text:
            empty_files.append(str(label_path))
            continue
        for line in text.splitlines():
            parts = line.split()
            if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                bad_files.append(str(label_path))
                continue
            cls_id = int(float(parts[0]))
            if cls_id < 0 or cls_id >= len(TARGET_LABELS):
                bad_files.append(str(label_path))
            else:
                class_counts[TARGET_LABELS[cls_id]] += 1

print("class_counts:", dict(class_counts))
print("empty_files:", len(empty_files))
print("bad_files:", len(set(bad_files)))
assert not bad_files, f"Invalid YOLO segmentation labels found: {bad_files[:5]}"
assert not empty_files, f"Empty label files found: {empty_files[:5]}"

In [ ]:
# 8) Visualize merged samples
COLORS = ["red", "blue", "green", "orange", "purple", "cyan", "magenta", "yellow"]


def draw_yolo_seg_sample(split="train", n=6):
    image_paths = [p for p in (OUTPUT_DIR / "images" / split).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    sample_paths = random.sample(image_paths, min(n, len(image_paths)))
    cols = 3
    rows = math.ceil(len(sample_paths) / cols)
    plt.figure(figsize=(16, 5 * rows))

    for idx, img_path in enumerate(sample_paths, 1):
        rel = img_path.relative_to(OUTPUT_DIR / "images" / split)
        label_path = (OUTPUT_DIR / "labels" / split / rel).with_suffix(".txt")
        image = Image.open(img_path).convert("RGB")
        w, h = image.size
        ax = plt.subplot(rows, cols, idx)
        ax.imshow(image)
        ax.set_title(f"{split}/{rel}")
        ax.axis("off")

        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) < 7:
                continue
            cls_id = int(float(parts[0]))
            coords = [float(v) for v in parts[1:]]
            xy = [(coords[i] * w, coords[i + 1] * h) for i in range(0, len(coords), 2)]
            color = COLORS[cls_id % len(COLORS)]
            ax.add_patch(patches.Polygon(xy, closed=True, fill=False, edgecolor=color, linewidth=2))
            if xy:
                ax.text(xy[0][0], xy[0][1], TARGET_LABELS[cls_id], color="white", fontsize=8,
                        bbox={"facecolor": color, "alpha": 0.7, "pad": 1})
    plt.tight_layout()
    plt.show()


draw_yolo_seg_sample("train", n=6)

In [ ]:
# 9) Configure MLflow
tracking_uri = MLRUNS_DIR.resolve().as_uri()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
mlflow.set_experiment(EXPERIMENT_NAME)
print("MLflow tracking URI:", tracking_uri)
print("Experiment:", EXPERIMENT_NAME)

In [ ]:
# 10) Optuna tuning + final training
def metric_value(metrics_obj, preferred="mask_map50_95") -> float:
    results = getattr(metrics_obj, "results_dict", None)
    if not isinstance(results, dict):
        results = metrics_obj if isinstance(metrics_obj, dict) else {}
    aliases = {
        "mask_map50_95": ["metrics/mAP50-95(M)", "metrics/mAP50-95(Mask)", "metrics/mAP50-95(B)"],
        "mask_map50": ["metrics/mAP50(M)", "metrics/mAP50(Mask)", "metrics/mAP50(B)"],
        "box_map50_95": ["metrics/mAP50-95(B)", "metrics/mAP50-95(M)"],
        "box_map50": ["metrics/mAP50(B)", "metrics/mAP50(M)"],
    }
    for key in aliases.get(preferred, [preferred]):
        if key in results:
            return float(results[key])
    for key, value in results.items():
        if preferred.lower() in str(key).lower():
            return float(value)
    return 0.0


def metrics_to_dict(metrics_obj) -> dict[str, Any]:
    results = getattr(metrics_obj, "results_dict", None)
    if isinstance(results, dict):
        return {str(k): float(v) if isinstance(v, (int, float)) else str(v) for k, v in results.items()}
    if isinstance(metrics_obj, dict):
        return dict(metrics_obj)
    return {"value": str(metrics_obj)}


def yolo_train_args(run_name: str, epochs: int, overrides: dict[str, Any] | None = None):
    args = {
        "data": str(dataset_yaml_path),
        "epochs": epochs,
        "batch": BATCH,
        "imgsz": IMGSZ,
        "patience": PATIENCE,
        "device": DEVICE,
        "project": str(RUNS_DIR),
        "name": run_name,
        "augment": True,
        "mosaic": 1.0,
        "mixup": 0.1,
        "copy_paste": 0.3,
        "optimizer": "AdamW",
        "lr0": 0.001,
        "lrf": 0.01,
        "weight_decay": 0.0005,
    }
    if overrides:
        args.update(overrides)
    return args


best_params = {}
start_time = time.time()

with mlflow.start_run(run_name=RUN_NAME) as parent_run:
    mlflow.log_params({
        "model": YOLO_MODEL,
        "dataset_yaml": str(dataset_yaml_path),
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "batch": BATCH,
        "device": DEVICE,
        "optuna_trials": 0 if SKIP_OPTUNA else OPTUNA_TRIALS,
        "tune_epochs": TUNE_EPOCHS,
        "seed": SEED,
        "classes": ",".join(TARGET_LABELS),
    })
    mlflow.log_artifact(str(dataset_yaml_path))
    mlflow.log_artifact(str(OUTPUT_DIR / "merge_summary.json"))
    taco_download_summary = DATA_DIR / "taco_download_summary.json"
    if taco_download_summary.exists():
        mlflow.log_artifact(str(taco_download_summary))

    if not SKIP_OPTUNA and OPTUNA_TRIALS > 0:
        def objective(trial: optuna.Trial) -> float:
            params = {
                "lr0": trial.suggest_float("lr0", 1e-4, 5e-2, log=True),
                "lrf": trial.suggest_float("lrf", 0.01, 0.5, log=True),
                "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
                "mosaic": trial.suggest_float("mosaic", 0.0, 1.0),
                "mixup": trial.suggest_float("mixup", 0.0, 0.3),
                "copy_paste": trial.suggest_float("copy_paste", 0.0, 0.5),
            }
            trial_run_name = f"{RUN_NAME}-trial-{trial.number:03d}"
            with mlflow.start_run(run_name=trial_run_name, nested=True):
                mlflow.log_params(params)
                model = YOLO(YOLO_MODEL)
                model.train(**yolo_train_args(trial_run_name, TUNE_EPOCHS, params))
                val_metrics = model.val(data=str(dataset_yaml_path), split="val", imgsz=IMGSZ)
                score = metric_value(val_metrics, "mask_map50_95")
                mlflow.log_metric("val_mask_map50_95", score)
                return score

        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=OPTUNA_TRIALS)
        best_params = dict(study.best_params)
        mlflow.log_metric("best_optuna_val_mask_map50_95", float(study.best_value))
        mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})
        (ARTIFACTS_DIR / "optuna_best_params.json").write_text(json.dumps(best_params, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(ARTIFACTS_DIR / "optuna_best_params.json"))

    print("Best params:", best_params)
    final_model = YOLO(YOLO_MODEL)
    final_model.train(**yolo_train_args(RUN_NAME, EPOCHS, best_params))

    trained_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
    if not trained_weights.exists():
        raise FileNotFoundError(f"Expected best weights were not found: {trained_weights}")

    mlflow.log_artifact(str(trained_weights), artifact_path="weights")
    mlflow.log_metric("training_elapsed_seconds", time.time() - start_time)

print("trained_weights:", trained_weights)

In [ ]:
# 11) Evaluate the best model on validation and test splits
best_model = YOLO(str(trained_weights))
val_metrics = best_model.val(data=str(dataset_yaml_path), split="val", imgsz=IMGSZ)
test_metrics = best_model.val(data=str(dataset_yaml_path), split="test", imgsz=IMGSZ)

val_dict = metrics_to_dict(val_metrics)
test_dict = metrics_to_dict(test_metrics)

metrics_summary = {
    "model": YOLO_MODEL,
    "trained_weights": str(trained_weights),
    "dataset_yaml": str(dataset_yaml_path),
    "best_params": best_params,
    "val_metrics": val_dict,
    "test_metrics": test_dict,
}

metrics_json_path = ARTIFACTS_DIR / "metrics_summary.json"
metrics_json_path.write_text(json.dumps(metrics_summary, indent=2), encoding="utf-8")

rows = []
for split_name, metric_dict in [("val", val_dict), ("test", test_dict)]:
    for key, value in metric_dict.items():
        rows.append({"split": split_name, "metric": key, "value": value})
metrics_csv_path = ARTIFACTS_DIR / "metrics_summary.csv"
pd.DataFrame(rows).to_csv(metrics_csv_path, index=False)

with mlflow.start_run(run_name=f"{RUN_NAME}-evaluation"):
    mlflow.log_params({"model": YOLO_MODEL, "weights": str(trained_weights), "dataset_yaml": str(dataset_yaml_path)})
    for key, value in val_dict.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric("val_" + key.replace("/", "_").replace("(", "").replace(")", ""), float(value))
    for key, value in test_dict.items():
        if isinstance(value, (int, float)):
            mlflow.log_metric("test_" + key.replace("/", "_").replace("(", "").replace(")", ""), float(value))
    mlflow.log_artifact(str(metrics_json_path))
    mlflow.log_artifact(str(metrics_csv_path))

print(json.dumps(metrics_summary, indent=2)[:4000])

In [ ]:
# 12) Copy/export the best model artifacts
best_weights_copy = ARTIFACTS_DIR / f"{RUN_NAME}_best.pt"
shutil.copy2(trained_weights, best_weights_copy)

exported_onnx = None
try:
    exported_path = Path(best_model.export(format="onnx", imgsz=IMGSZ, simplify=True))
    exported_onnx = ARTIFACTS_DIR / exported_path.name
    shutil.copy2(exported_path, exported_onnx)
except Exception as exc:
    print("ONNX export failed. Keeping .pt weights only.")
    print(type(exc).__name__, exc)

artifact_summary = {
    "best_weights": str(best_weights_copy),
    "exported_onnx": str(exported_onnx) if exported_onnx else None,
    "metrics_json": str(metrics_json_path),
    "metrics_csv": str(metrics_csv_path),
    "dataset_yaml": str(dataset_yaml_path),
    "mlruns": str(MLRUNS_DIR),
}
artifact_summary_path = ARTIFACTS_DIR / "artifact_summary.json"
artifact_summary_path.write_text(json.dumps(artifact_summary, indent=2), encoding="utf-8")

print(json.dumps(artifact_summary, indent=2))

In [ ]:
# 13) Test best model visually on held-out test images
sample_test_images = [p for p in (OUTPUT_DIR / "images" / "test").rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
sample_test_images = random.sample(sample_test_images, min(8, len(sample_test_images)))

predictions = best_model.predict(
    source=[str(p) for p in sample_test_images],
    imgsz=IMGSZ,
    conf=0.25,
    save=True,
    project=str(ARTIFACTS_DIR),
    name="test_predictions",
)

print("Saved prediction visualizations under:", ARTIFACTS_DIR / "test_predictions")
print("Predicted images:", len(predictions))